# Cypher 내보내기 점검

합성 CSV 입력을 만들고 `cypher_export`가 생성한 Browser용 Cypher 앞부분을 확인합니다.

In [ ]:
from pathlib import Path
import csv
import json
import tempfile

WORK = Path(tempfile.mkdtemp())
CSV_DIR = WORK / "csv"
OUTPUT = WORK / "load.cypher"
CSV_DIR.mkdir()


## 합성 입력 예시

In [ ]:
def write_csv(name, columns, rows):
    with (CSV_DIR / name).open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)

write_csv("pf_load_runs.csv", ["run_id", "input_folder_name", "model_name", "started_at", "node_count", "edge_count"], [])
write_csv("pf_nodes.csv", ["id", "node_type", "props", "chunk_id", "run_id"], [{"id": "benefit-1", "node_type": "Benefit", "props": json.dumps({"name": "합성 커피 혜택", "tags": ["커피"]}, ensure_ascii=False), "chunk_id": "sample", "run_id": "sample-run"}])
write_csv("pf_edges.csv", ["id", "source_id", "target_id", "edge_type", "run_id"], [])
write_csv("pf_evidence.csv", ["node_id", "edge_id", "doc_sha256", "page_no", "start_pos", "end_pos", "quote_status"], [])


## 생성 결과 앞부분

In [ ]:
from payment_card.file_pipeline.graph.cypher_export import export_cypher

summary = export_cypher(CSV_DIR, OUTPUT)
print(summary)
print("\n".join(OUTPUT.read_text(encoding="utf-8").splitlines()[:12]))
